In [1]:
import re
import urllib.request
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Qurilma:", device)

# 1. Ma'lumotlarni tayyyorlash
URL = "https://github.com/laxmimerit/IMDB-Movie-Reviews-Large-Dataset-50k/raw/master/train.xlsx"
urllib.request.urlretrieve(URL, "train.xlsx")

df = pd.read_excel("train.xlsx")
df = df.rename(columns={"Reviews": "review", "Sentiment": "sentiment"})[["review", "sentiment"]]
df["sentiment"] = df["sentiment"].map({"pos": "positive", "neg": "negative"})

pos = df[df["sentiment"] == "positive"].sample(1000, random_state=101)
neg = df[df["sentiment"] == "negative"].sample(1000, random_state=101)
data = pd.concat([pos, neg]).sample(frac=1, random_state=101).reset_index(drop=True)
print("Ma'lumot:", data.shape, data["sentiment"].value_counts().to_dict())

# 2. RNN uchun kodlash
def tokenize(text):
    text = re.sub(r"<.*?>", " ", text)
    return re.findall(r"\b\w+\b", text.lower())

counter = Counter(w for r in data["review"] for w in tokenize(r))
vocab = {"<pad>": 0, "<unk>": 1}
for i, (word, _) in enumerate(counter.most_common(4000), start=2):
    vocab[word] = i
print("Lug'at hajmi:", len(vocab))

MAX_LEN = 256

def encode_review(text, max_len=MAX_LEN):
    ids = [vocab.get(w, 1) for w in tokenize(text)][:max_len]
    return [0] * (max_len - len(ids)) + ids

data["encoded"] = data["review"].apply(encode_review)

# 3. SentimentRNN
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        _, h = self.rnn(self.embedding(x))
        return self.sigmoid(self.fc(h[-1])).squeeze(1)

X = torch.tensor(data["encoded"].tolist())
y = torch.tensor((data["sentiment"] == "positive").astype(float).values, dtype=torch.float32)
loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)

rnn_model = SentimentRNN(len(vocab)).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(rnn_model.parameters(), lr=0.001)

print("\n--- RNN o'qitish ---")
for epoch in range(10):
    rnn_model.train()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(rnn_model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(rnn_model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    print(f"Epoxa {epoch + 1}/10 | o'rtacha loss: {total / len(loader):.4f}")

# 4. Trasnferlar uchun ma'lumotlar
from datasets import Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          Trainer, TrainingArguments)

hf_data = data[["review", "sentiment"]].copy()
hf_data["label"] = (hf_data["sentiment"] == "positive").astype(int)
dataset = Dataset.from_pandas(hf_data[["review", "label"]])

split = dataset.train_test_split(test_size=500, seed=123)
train_ds, test_ds = split["train"], split["test"]

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
    return tokenizer(examples["review"], padding="max_length", truncation=True, max_length=256)

train_ds = train_ds.map(preprocess_function, batched=True)
test_ds = test_ds.map(preprocess_function, batched=True)

# 5. Transfer modelini o'qitish va baholash
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    logging_steps=100,
    per_device_train_batch_size=16,
    report_to="none",
)
trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=test_ds)

print("\n--- Transformer o'qitish ---")
trainer.train()
eval_results = trainer.evaluate()
print("Baholash natijalari:", eval_results)

# 6. Bashorat va taqqoshlash
def predict_rnn(text):
    rnn_model.eval()
    with torch.no_grad():
        prob = rnn_model(torch.tensor([encode_review(text)]).to(device)).item()
    return "positive" if prob >= 0.5 else "negative"

def predict_transformer(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    with torch.no_grad():
        pred = model(**inputs).logits.argmax(dim=-1).item()
    return model.config.id2label[pred]

reviews = {
    "a) Bu filmni juda uzoq kutgandim, lekin umuman yoqmadi.":
        "I had been waiting for this movie for so long, but I didn't like it at all.",
    "b) Aktyorlar jamoasi ajoyib, syujet ham juda qiziqarli ekan.":
        "The cast is wonderful, and the plot turned out to be very interesting.",
    "c) Filmning vizual effektlari yaxshi, ammo hikoyasi zaif.":
        "The visual effects of the movie are good, but the story is weak.",
}

print("\n--- Bashoratlar ---")
for uz, en in reviews.items():
    print(uz)
    print("  O'zbekcha -> RNN:", predict_rnn(uz), "| Transformer:", predict_transformer(uz))
    print("  Inglizcha -> RNN:", predict_rnn(en), "| Transformer:", predict_transformer(en))

"""
Xulosa:
 RNN o'qitish ma'lumotida loss'ni 0.71 dan 0.06 gacha tushirdi, Transformer esa ko'rilmagan
 500 ta test sharhida eval_loss = 0.425 ko'rsatdi. Inglizcha sinov sharhlarida ikkala model
 a) va b) sharhlarni to'g'ri aniqladi, aralash fikrli c) sharhda RNN "positive", Transformer
 "negative" dedi (so'zdan keyingi "hikoyasi zaif" qismi hukmni belgilagani uchun Transformerning
 javobi mantiqiyroq). Shu sababli bu vazifa uchun Transformer (DistilBERT) yaxshiroq: oldindan
 o'qitilgan, kontekstni tushunadi, RNN esa faqat 2000 ta sharhdan o'rgangani uchun ishonchsizroq.
-Eslatma: modellar inglizcha ma'lumotda o'qitilgani uchun o'zbekcha sharhlarda ikkalasi ham
 hamma matnni "positive" deb belgiladi (a) noto'g'ri), ya'ni o'zbek tilini tushunmaydi.
"""

Qurilma: cuda
Ma'lumot: (2000, 2) {'negative': 1000, 'positive': 1000}
Lug'at hajmi: 4002

--- RNN o'qitish ---
Epoxa 1/10 | o'rtacha loss: 0.7093
Epoxa 2/10 | o'rtacha loss: 0.6384
Epoxa 3/10 | o'rtacha loss: 0.5671
Epoxa 4/10 | o'rtacha loss: 0.4761
Epoxa 5/10 | o'rtacha loss: 0.3924
Epoxa 6/10 | o'rtacha loss: 0.2843
Epoxa 7/10 | o'rtacha loss: 0.1835
Epoxa 8/10 | o'rtacha loss: 0.1072
Epoxa 9/10 | o'rtacha loss: 0.0795
Epoxa 10/10 | o'rtacha loss: 0.0592


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...


--- Transformer o'qitish ---


Step,Training Loss
100,0.431747
200,0.218759


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Step
0.218759,0.425237,282


Baholash natijalari: {'eval_loss': 0.42523735761642456}

--- Bashoratlar ---
a) Bu filmni juda uzoq kutgandim, lekin umuman yoqmadi.
  O'zbekcha -> RNN: positive | Transformer: positive
  Inglizcha -> RNN: negative | Transformer: negative
b) Aktyorlar jamoasi ajoyib, syujet ham juda qiziqarli ekan.
  O'zbekcha -> RNN: positive | Transformer: positive
  Inglizcha -> RNN: positive | Transformer: positive
c) Filmning vizual effektlari yaxshi, ammo hikoyasi zaif.
  O'zbekcha -> RNN: positive | Transformer: positive
  Inglizcha -> RNN: positive | Transformer: negative
